In [1]:
import pandas as pd
import glob

files = glob.glob("../data/raw/*.xlsx")
print(files)

test = pd.read_excel(files[0])
print(test.head())
print(test.dtypes)

[]


IndexError: list index out of range

In [2]:
import os
print(os.getcwd())
print(os.listdir("../data/raw"))

c:\Users\BoringSakshi\OneDrive\Desktop\UPI Project\data\clean


FileNotFoundError: [WinError 3] The system cannot find the path specified: '../data/raw'

In [ ]:
import pandas as pd
import glob

files = glob.glob("../data/raw/*.xlsx")
print(len(files))
print(files)

test = pd.read_excel(files[0])
print(test.head())
print(test.dtypes)

In [1]:
import pandas as pd
import glob

files = glob.glob("../data/raw/*.xlsx")
print(len(files))
print(files)

test = pd.read_excel(files[0])
print(test.head())
print(test.dtypes)

8
['../data/raw\\Product-Statistics-UPI-Product-statistics-upi-2019-20-Monthly.xlsx', '../data/raw\\Product-Statistics-UPI-Product-statistics-upi-2020-21-Monthly.xlsx', '../data/raw\\Product-Statistics-UPI-Product-statistics-upi-2021-22-Monthly.xlsx', '../data/raw\\Product-Statistics-UPI-Product-statistics-upi-2022-23-Monthly.xlsx', '../data/raw\\Product-Statistics-UPI-Product-statistics-upi-2023-24-Monthly.xlsx', '../data/raw\\Product-Statistics-UPI-Product-statistics-upi-2024-25-Monthly.xlsx', '../data/raw\\Product-Statistics-UPI-Product-statistics-upi-2025-26-Monthly.xlsx', '../data/raw\\Product-Statistics-UPI-Product-statistics-upi-2026-27-Monthly.xlsx']
           Month  No. of Banks live on UPI  Volume (In Mn.) Value (In Cr.)
0     March-2020                       148          1246.84    2,06,462.31
1  February-2020                       146          1325.69    2,22,516.95
2   January-2020                       144          1305.02    2,16,242.97
3  December-2019                 

In [2]:
# 1. Read all 8 files and stack them into one table
frames = [pd.read_excel(f) for f in files]
df = pd.concat(frames, ignore_index=True)

# 2. Simple column names
df.columns = ["month", "banks_live", "volume_mn", "value_cr"]

# 3. Remove commas from value and convert to a number
df["value_cr"] = df["value_cr"].astype(str).str.replace(",", "").astype(float)

# 4. Convert month text to a real date
df["month"] = pd.to_datetime(df["month"], format="%B-%Y")

# 5. Remove duplicates, sort oldest to newest
df = df.drop_duplicates().sort_values("month").reset_index(drop=True)

print(df.shape)
print(df.head())
print(df.tail())

(90, 4)
       month  banks_live volume_mn   value_cr
0 2019-04-01         144    781.79  142034.39
1 2019-05-01         143    733.54  152449.29
2 2019-06-01         142    754.54  146566.35
3 2019-07-01         143    822.29  146386.64
4 2019-08-01         141    918.35  154504.89
        month  banks_live  volume_mn    value_cr
85 2026-05-01         720   23201.93  2990424.21
86 2026-06-01         731   22716.07  2892138.67
87 2026-07-01         741  23,658.35  2987880.49
88 2026-08-01         752  24,508.96  2982355.95
89 2026-09-01         756  24,068.38  2937396.67


In [3]:
print(df.isna().sum())                      # missing values?
print(df["month"].diff().dt.days.value_counts())   # gaps between months (should be 28-31 only)
print(df["month"].min(), df["month"].max())

month         0
banks_live    0
volume_mn     0
value_cr      0
dtype: int64
month
31.0    52
30.0    30
28.0     5
29.0     2
Name: count, dtype: int64
2019-04-01 00:00:00 2026-09-01 00:00:00


In [4]:
# Days in each month, so we can compare months fairly
df["days_in_month"] = df["month"].dt.days_in_month

# Average daily volume (millions per day)
df["avg_daily_volume_mn"] = (df["volume_mn"] / df["days_in_month"]).round(2)

# Average payment size in rupees (value in crore / volume in million x 10)
df["avg_ticket_rs"] = (df["value_cr"] / df["volume_mn"] * 10).round(2)

print(df.head())
print(df.tail())

df.to_csv("../data/clean/monthly_upi.csv", index=False)

TypeError: unsupported operand type(s) for /: 'str' and 'int'

In [5]:
# Days in each month, so we can compare months fairly
df["days_in_month"] = df["month"].dt.days_in_month

# Average daily volume (millions per day)
df["avg_daily_volume_mn"] = (df["volume_mn"] / df["days_in_month"]).round(2)

# Average payment size in rupees (value in crore / volume in million x 10)
df["avg_ticket_rs"] = (df["value_cr"] / df["volume_mn"] * 10).round(2)

print(df.head())
print(df.tail())

df.to_csv("../data/clean/monthly_upi.csv", index=False)

TypeError: unsupported operand type(s) for /: 'str' and 'int'

In [7]:
df["volume_mn"] = df["volume_mn"].astype(str).str.replace(",", "").astype(float)

print(df.dtypes)

month            datetime64[us]
banks_live                int64
volume_mn               float64
value_cr                float64
days_in_month             int32
dtype: object


In [8]:
df["days_in_month"] = df["month"].dt.days_in_month
df["avg_daily_volume_mn"] = (df["volume_mn"] / df["days_in_month"]).round(2)
df["avg_ticket_rs"] = (df["value_cr"] / df["volume_mn"] * 10).round(2)

print(df.shape)
print(df.head())
print(df.tail())

df.to_csv("../data/clean/monthly_upi.csv", index=False)

(90, 7)
       month  banks_live  volume_mn   value_cr  days_in_month  \
0 2019-04-01         144     781.79  142034.39             30   
1 2019-05-01         143     733.54  152449.29             31   
2 2019-06-01         142     754.54  146566.35             30   
3 2019-07-01         143     822.29  146386.64             31   
4 2019-08-01         141     918.35  154504.89             31   

   avg_daily_volume_mn  avg_ticket_rs  
0                26.06        1816.78  
1                23.66        2078.27  
2                25.15        1942.46  
3                26.53        1780.23  
4                29.62        1682.42  
        month  banks_live  volume_mn    value_cr  days_in_month  \
85 2026-05-01         720   23201.93  2990424.21             31   
86 2026-06-01         731   22716.07  2892138.67             30   
87 2026-07-01         741   23658.35  2987880.49             31   
88 2026-08-01         752   24508.96  2982355.95             31   
89 2026-09-01         756 